# LangChain 객체 직렬화


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


여기서 직렬화하는 것은 **모델 가중치가 아니라 LangChain 구성 객체**입니다. 재현 가능한 배포에는 직렬화 파일과 함께 의존성 버전 및 환경 변수 이름도 관리해야 합니다.

신뢰할 수 없는 pickle은 임의 코드 실행 위험이 있으므로 이 예제에서는 사용하지 않고 사람이 검토 가능한 JSON만 사용합니다.


In [ ]:
%pip install -qU langchain-core langchain-openai python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

load_dotenv()

prompt = PromptTemplate.from_template("{fruit}의 대표적인 색상은 무엇입니까?")
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    temperature=0,
)
chain = prompt | llm | StrOutputParser()

print("PromptTemplate:", prompt.is_lc_serializable())
print("ChatOpenAI:", llm.is_lc_serializable())
print("chain:", chain.is_lc_serializable())


## `dumpd`와 `dumps`


In [ ]:
from langchain_core.load import dumpd, dumps

chain_dict = dumpd(chain)
chain_json = dumps(chain, pretty=True)

print(type(chain_dict), type(chain_json))
print(chain_json[:1000])


직렬화 결과에는 실제 API 키 대신 secret 식별자가 기록됩니다. JSON 파일에는 여전히 엔드포인트·모델명 같은 설정이 들어갈 수 있으므로 공유 전에 검토합니다.


In [ ]:
from pathlib import Path

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
json_path = artifact_dir / "fruit_chain.json"
json_path.write_text(chain_json, encoding="utf-8")
print(json_path.resolve())


## 안전하게 다시 로드하기

직렬화 파일은 직접 만들었거나 출처를 신뢰할 수 있는 경우에만 로드합니다. secret 값은 런타임 환경에서 명시적으로 주입합니다.


In [ ]:
from langchain_core.load import loads

serialized = json_path.read_text(encoding="utf-8")
restored_chain = loads(
    serialized,
    secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]},
)

print(restored_chain.invoke({"fruit": "사과"}))


### 운영 시 주의점

- JSON은 코드·패키지 버전의 장기 호환성을 보장하는 모델 포맷이 아닙니다.
- 배포에는 잠금 파일(`uv.lock`, `poetry.lock`, 고정된 requirements 등)을 함께 보관합니다.
- 외부에서 받은 pickle과 출처가 불명확한 LangChain 직렬화 파일을 로드하지 않습니다.
- 장기 보관이 목적이면 작은 JSON/YAML 설정으로 직접 모델을 재구성하는 방식이 더 명시적입니다.
